# Issue #399 — #394 close-out addendum

Contract: `BJ-UNIFIED-N7-MIXED-CLOSEOUT-001/v1`. Terminal decision is unchanged: `NO_MIXED_N7_EXTENSION_INCREMENT`. This notebook regenerates the two corrected statistics from the frozen #394 artifacts and validates the committed addendum package (E06/E08).

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from src.n7_mixed_full_history import closeout as C
from src.n7_mixed_full_history.pipeline import _matched_contrasts_board, _paired_frame_for_schedule
REPO=Path.cwd()
SRC=REPO/'results/n7_mixed_full_history'
ADD=REPO/'results/n7_mixed_full_history_addendum'
bundles=pd.read_parquet(SRC/'bundles.parquet')
frozen=pd.read_csv(SRC/'matched_contrasts.csv')
C.assert_frozen_board_parity(bundles, frozen)
print('frozen board parity OK')
decision=json.loads((SRC/'decision.json').read_text())
decision

frozen board parity OK


{'consumes_218': False,
 'contract': 'BJ-UNIFIED-N7-MIXED-FULL-HISTORY-001/v1',
 'evidence_cutoff_ts': '2026-08-25T14:10:00Z',
 'exposed_historical_only': True,
 'failed_predicates': ['R3_2025_positive',
  'C1E3_2025_positive',
  'R3_extension_positive',
  'R3_extension_ex_largest_positive',
  'C1E3_extension_ex_largest_positive',
  'mixed_extension_positive',
  'mixed_extension_ex_largest_positive',
  'mixed_maxdd_below_each_standalone',
  'diversification_not_mere_dilution',
  'no_annual_extension_year_reversal',
  'moving_block_p05_vs_r3_ext_nonnegative',
  'moving_block_p05_vs_c1e3_ext_nonnegative',
  'moving_block_p05_vs_r3_all_nonnegative',
  'moving_block_p05_vs_c1e3_all_nonnegative',
  'extension_confirms_pre2025_r3'],
 'predictates_report': {'C1E3_2025_positive': False,
  'C1E3_2026_positive': True,
  'C1E3_all_positive': True,
  'C1E3_extension_ex_largest_positive': False,
  'C1E3_extension_positive': True,
  'R3_2025_positive': False,
  'R3_2026_positive': True,
  'R3_all_po

In [2]:
corrected, paired_frame, counters = C.corrected_contrasts(bundles)
assert int(counters['differ'].sum()) >= 1
assert int(counters['sign_change'].sum()) >= 1
corrected

,scope,subject_schedule,reference_schedule,contrast,issues,subject_pnl_cny,reference_pnl_cny,matched_lift_cny,subject_ex_largest_pnl_cny,reference_ex_largest_pnl_cny,subject_largest_payout_issue,reference_largest_payout_issue,armwise_ex_largest_lift_cny,largest_abs_paired_delta_cny,largest_abs_paired_delta_signed_cny,largest_abs_paired_delta_issue,largest_absolute_paired_delta_sensitivity_cny
0,PRE_2025,MIXED_50_50,R3_STANDALONE,vs_R3,151,52666.797417,54671.399779,-2004.602362,23221.606361,-5242.982332,23021,23021,28464.588693,30469.191055,-30469.191055,23021,28464.588693
1,PRE_2025,MIXED_50_50,C1E3_STANDALONE,vs_C1E3,151,52666.797417,46228.300875,6438.496541,23221.606361,9262.746129,23021,23055,13958.860232,30469.191055,30469.191055,23021,-24030.694514
2,PRE_2025,MIXED_50_50,R0_COMPARATOR,vs_R0,151,52666.797417,-13729.938991,66396.736408,23221.606361,-26537.418636,23021,24043,49759.024997,30469.191055,30469.191055,23021,35927.545352
3,EXTENSION_2025_2026,MIXED_50_50,R3_STANDALONE,vs_R3,117,-1098.951725,-4304.027854,3205.076129,-14174.268780,-22079.117260,26086,26086,7904.848481,8830.000104,8830.000104,25033,-5624.923975
4,EXTENSION_2025_2026,MIXED_50_50,C1E3_STANDALONE,vs_C1E3,117,-1098.951725,2535.218222,-3634.169947,-14174.268780,-10540.098833,26086,26086,-3634.169947,5587.841477,5587.841477,25114,-9222.011424
5,EXTENSION_2025_2026,MIXED_50_50,R0_COMPARATOR,vs_R0,117,-1098.951725,30030.789268,-31129.740993,-14174.268780,9894.552551,26086,26056,-24068.821331,21160.236717,-21160.236717,26056,-9969.504276
6,ALL,MIXED_50_50,R3_STANDALONE,vs_R3,268,51567.845692,50367.371925,1200.473767,22122.654636,-9547.010186,23021,23021,31669.664822,30469.191055,-30469.191055,23021,31669.664822
7,ALL,MIXED_50_50,C1E3_STANDALONE,vs_C1E3,268,51567.845692,48763.519098,2804.326594,22122.654636,11797.964352,23021,23055,10324.690285,30469.191055,30469.191055,23021,-27664.864461
8,ALL,MIXED_50_50,R0_COMPARATOR,vs_R0,268,51567.845692,16300.850277,35266.995415,22122.654636,-3835.386440,23021,26056,25958.041076,30469.191055,30469.191055,23021,4797.804359


In [3]:
counters

,scope,contrast,matched_lift_cny,armwise_ex_largest_lift_cny,largest_absolute_paired_delta_sensitivity_cny,differ,sign_change,role
0,PRE_2025,vs_R3,-2004.602362,28464.588693,28464.588693,False,False,coincident_equal
1,PRE_2025,vs_C1E3,6438.496541,13958.860232,-24030.694514,True,True,sign_changing_regression
2,PRE_2025,vs_R0,66396.736408,49759.024997,35927.545352,True,False,counterexample_differ
3,EXTENSION_2025_2026,vs_R3,3205.076129,7904.848481,-5624.923975,True,True,sign_changing_regression
4,EXTENSION_2025_2026,vs_C1E3,-3634.169947,-3634.169947,-9222.011424,True,False,counterexample_differ
5,EXTENSION_2025_2026,vs_R0,-31129.740993,-24068.821331,-9969.504276,True,False,counterexample_differ
6,ALL,vs_R3,1200.473767,31669.664822,31669.664822,False,False,coincident_equal
7,ALL,vs_C1E3,2804.326594,10324.690285,-27664.864461,True,True,sign_changing_regression
8,ALL,vs_R0,35266.995415,25958.041076,4797.804359,True,False,counterexample_differ


In [4]:
assert (ADD/'matched_contrasts_corrected.csv').is_file()
committed = pd.read_csv(ADD/'matched_contrasts_corrected.csv')
keys = ['scope','contrast']
cmp = committed.merge(corrected, on=keys, suffixes=('_committed','_recomputed'), how='outer')
fcols = [c for c in cmp.columns if not c.endswith(('_committed','_recomputed')) or c in keys]
numcols = [c for c in ['subject_pnl_cny','reference_pnl_cny','matched_lift_cny','subject_ex_largest_pnl_cny','reference_ex_largest_pnl_cny','armwise_ex_largest_lift_cny','largest_abs_paired_delta_cny','largest_abs_paired_delta_signed_cny','largest_absolute_paired_delta_sensitivity_cny']]
for c in numcols:
    a = cmp[c+'_committed'].to_numpy(float)
    b = cmp[c+'_recomputed'].to_numpy(float)
    assert np.allclose(a, b, rtol=1e-9, atol=1e-9), c
strcols = ['subject_largest_payout_issue','reference_largest_payout_issue','largest_abs_paired_delta_issue']
for c in strcols:
    assert (cmp[c+'_committed'].astype(str) == cmp[c+'_recomputed'].astype(str)).all(), c
assert (cmp['subject_schedule_committed'].astype(str) == 'MIXED_50_50').all()
assert cmp['reference_schedule_committed'].isin(['R3_STANDALONE','C1E3_STANDALONE','R0_COMPARATOR']).all()
receipt=json.loads((ADD/'ADDENDUM_RECEIPT.json').read_text())
assert receipt['source']['source_merge_sha'] == C.SOURCE_MERGE_SHA
print('committed addendum package parity OK')
receipt['deprecation']

committed addendum package parity OK


{'deprecated_field': 'matched_ex_largest_lift_cny',
 'merged_value_semantics': 'delta.sum() - delta[argmax(abs(delta))] on the exact-common issue set (paired-delta sensitivity), not armwise ex-largest',
 'statement': 'The merged #394 field `matched_ex_largest_lift_cny` is DEPRECATED and MUST NOT be consumed as an armwise ex-largest lift. It was computed as the paired-delta sensitivity `delta.sum() - delta[argmax(abs(delta))]` on the exact-common issue set, not as an armwise ex-largest removal. Consumers must use `armwise_ex_largest_lift_cny` for armwise ex-largest semantics and `largest_absolute_paired_delta_sensitivity_cny` for the paired-delta sensitivity semantics.',
 'status': 'DEPRECATED'}

In [5]:
mb=json.loads((ADD/'moving_block_metadata.json').read_text())
assert mb['estimand'] == 'paired_mean_cny_per_issue'
assert mb['unit'] == 'CNY_PER_ISSUE'
assert mb['block_length'] == 10 and mb['draw_count'] == 5000 and mb['seed'] == 394001
mb

{'block_length': 10,
 'draw_count': 5000,
 'estimand': 'paired_mean_cny_per_issue',
 'reconciles_to_producer_config': True,
 'rows': [{'contrast': 'vs_C1E3',
   'issues': 268,
   'moving_block_p05_lift_cny_per_issue': -273.7617736396171,
   'scope': 'ALL'},
  {'contrast': 'vs_R0',
   'issues': 268,
   'moving_block_p05_lift_cny_per_issue': -246.72890929203663,
   'scope': 'ALL'},
  {'contrast': 'vs_R3',
   'issues': 268,
   'moving_block_p05_lift_cny_per_issue': -328.6746792602363,
   'scope': 'ALL'},
  {'contrast': 'vs_C1E3',
   'issues': 117,
   'moving_block_p05_lift_cny_per_issue': -207.48987752945897,
   'scope': 'EXTENSION_2025_2026'},
  {'contrast': 'vs_R0',
   'issues': 117,
   'moving_block_p05_lift_cny_per_issue': -756.6615889609543,
   'scope': 'EXTENSION_2025_2026'},
  {'contrast': 'vs_R3',
   'issues': 117,
   'moving_block_p05_lift_cny_per_issue': -209.16697648375236,
   'scope': 'EXTENSION_2025_2026'},
  {'contrast': 'vs_C1E3',
   'issues': 151,
   'moving_block_p05_lift

In [6]:
dr=json.loads((ADD/'DECISION_RECEIPT.json').read_text())
assert dr['verdict_unchanged'] == 'NO_MIXED_N7_EXTENSION_INCREMENT'
print('decision invariance OK')
dr

decision invariance OK


{'contract': 'BJ-UNIFIED-N7-MIXED-CLOSEOUT-001/v1',
 'deprecated_field_never_decision_driving': 'the merged `matched_ex_largest_lift_cny` appears only in the descriptive matched-contrasts table of the frozen pipeline; the terminal decision gates used native-denominator `ex_largest_pnl_cny` and `moving_block_p05_lift_cny` only',
 'original_decision_file_sha256': '4e45f74a5ff717db8ba82c72cb7c649212aa773b8e9e61ea90bf1dc11f99db9c',
 'preserved_issues': [210, 218],
 'prospective_authorized_unchanged': False,
 'strategy_promoted_unchanged': False,
 'verdict_unchanged': 'NO_MIXED_N7_EXTENSION_INCREMENT'}

All values regenerated deterministically from the frozen #394 artifacts. The merged `matched_ex_largest_lift_cny` remains byte-unchanged in the frozen package and is DEPRECATED as an armwise ex-largest field.